# IEDB source-antigen overlap audit

Purpose: assess source-protein/parent-antigen overlap between one training FASTA and any number of test FASTAs. 

**No FASTA is modified, no model is trained, and no labels are replaced.** This notebook audits current IEDB metadata, not historical source-label validity. An exact sequence match for an external peptide yields candidate provenance, not proof of its original benchmark source. Missing mappings are never treated as evidence of independence.

## Run order
1. Upload FASTAs as a Dataset and attach it through **Add Input**. Enable Internet in notebook settings; CPU is sufficient. (Kaggle is recommended for this notebook execution)
2. Run cells through file selection and inspect the input summary.
3. Run the offline tests and **small live smoke test**. Inspect their output before proceeding.
4. Set `RUN_FULL_AUDIT = True` in the clearly marked full-run cell and run it, followed by report generation.
5. Download `iedb_antigen_audit_results.zip` from `/kaggle/working/iedb_antigen_audit`.

The full run is disabled by default, so **Run All initially stops after the smoke test**.

Cache survives reruns within a session. To resume in another session, save/download `iedb_cache.zip`, attach it as another Input, and set `RESTORE_CACHE_ZIP` below. Kaggle working storage does not automatically persist across fresh sessions.

Sources: [IEDB API documentation](https://discuss.iedb.org/t/immune-epitope-database-query-api-iq-api/154), [API schema](https://query-api.iedb.org/docs/swagger/). Retrieval date, settings, input SHA256 hashes and raw responses are saved for reproducibility.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, hashlib, json, re, time, urllib.request, urllib.parse, urllib.error, zipfile, shutil, sys, platform
from IPython.display import display, FileLink

API_URL = "https://query-api.iedb.org/epitope_search"
INPUT_ROOT = Path("/kaggle/input/datasets/_______/bcell-dataset") # Provide the main folder path of the dataset here
OUTPUT_DIR = Path("/kaggle/working/iedb_antigen_audit") if Path("/kaggle").exists() else Path.cwd() / "iedb_antigen_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)
ID_BATCH_SIZE = 100
SEQUENCE_BATCH_SIZE = 20
PAGE_SIZE = 1000
REQUEST_TIMEOUT = 45
MAX_RETRIES = 4
PAUSE_SECONDS = 0.5
REFRESH_CACHE = False
# To restore a previous run, specify the ZIP path in Kaggle Inputs, otherwise leave None.
RESTORE_CACHE_ZIP = None
CACHE_SCHEMA = "iedb-antigen-audit-v1"
FIELDS = "structure_id,linear_sequence,curated_source_antigens,parent_source_antigen_iris,parent_source_antigen_names"
REQUEST_LOG = []

def now(): return datetime.now(timezone.utc).isoformat()
def digest(text): return hashlib.sha256(text.encode()).hexdigest()
def atomic_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

if RESTORE_CACHE_ZIP:
    with zipfile.ZipFile(RESTORE_CACHE_ZIP) as z:
        for item in z.infolist():
            p = Path(item.filename)
            # Cache archives generated below contain only cache/*.json.
            if len(p.parts) == 2 and p.parts[0] == "cache" and p.suffix == ".json":
                (CACHE_DIR / p.name).write_bytes(z.read(item))
    print("Cache restored.")
print("Python:", sys.version.split()[0], "Outputs:", OUTPUT_DIR)


## Discover and select FASTAs
The dropdown chooses training; the multi-select chooses test versions. Use Ctrl/Cmd-click to select several tests. No ambiguous training-name match is selected automatically.

If widgets do not appear or do not respond in Kaggle, use the manual index/path cell immediately below. The inventory index is stable until discovery is rerun.


In [ ]:
FASTA_FILES = sorted(p for p in INPUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in {".fasta", ".fa", ".faa", ".fas", ".fsa"}) if INPUT_ROOT.exists() else []
if not FASTA_FILES:
    print("No FASTAs found. Attach a Kaggle Input dataset, rerun discovery, or enter manual paths below.")
for i, p in enumerate(FASTA_FILES): print(f"[{i}] {p}")

def canonical_name(p): return re.sub(r"^\d+-", "", p.name.lower())
train_matches = [str(p) for p in FASTA_FILES if canonical_name(p) == "train-ibce.fasta"]
DEFAULT_TRAIN = train_matches[0] if len(train_matches) == 1 else None
DEFAULT_TESTS = [
    str(p) for p in FASTA_FILES
    if canonical_name(p) in {
        "Int-ibce.fasta",  
        "int_ibce.fasta",
        "abcpred.fasta",
        "extI_clbe.fasta",
        "extII_abcpred.fasta",
    }
]
WIDGETS_AVAILABLE = False
try:
    import ipywidgets as widgets
    options = [(str(p.relative_to(INPUT_ROOT)), str(p)) for p in FASTA_FILES]
    train_widget = widgets.Dropdown(options=[("Choose training FASTA", None)] + options, value=DEFAULT_TRAIN, description="Training:", layout=widgets.Layout(width="95%"))
    test_widget = widgets.SelectMultiple(options=options, value=tuple(DEFAULT_TESTS), description="Tests:", rows=min(16, max(4, len(options))), layout=widgets.Layout(width="95%"))
    display(train_widget, test_widget)
    WIDGETS_AVAILABLE = True
except ImportError:
    print("ipywidgets is unavailable. Use manual indices/paths below; no installation is required.")


In [ ]:
# Leave these as None/[] to use widget choices. An index refers to the inventory above.
# Examples: MANUAL_TRAIN = 0; MANUAL_TESTS = [1, 2, 3]
# Or: MANUAL_TRAIN = "/kaggle/input/my-data/Train-ibce.fasta"
MANUAL_TRAIN = None
MANUAL_TESTS = []

def choose_path(value):
    if isinstance(value, int): return FASTA_FILES[value]
    return Path(value)

training_choice = MANUAL_TRAIN if MANUAL_TRAIN is not None else (train_widget.value if WIDGETS_AVAILABLE else DEFAULT_TRAIN)
test_choices = MANUAL_TESTS if MANUAL_TESTS else (list(test_widget.value) if WIDGETS_AVAILABLE else DEFAULT_TESTS)
if not training_choice or not test_choices:
    raise ValueError("Choose one training FASTA and at least one test FASTA, then rerun this cell.")
TRAIN_PATH = choose_path(training_choice).resolve()
TEST_PATHS = list(dict.fromkeys(choose_path(v).resolve() for v in test_choices))
if TRAIN_PATH in TEST_PATHS: raise ValueError("Training FASTA must not also be selected as a test.")
for p in [TRAIN_PATH] + TEST_PATHS:
    if not p.is_file(): raise FileNotFoundError(p)
print("Training:", TRAIN_PATH)
print("Tests:", *TEST_PATHS, sep="\n")


## Read inputs and recover known IEDB IDs
FASTA headers with `IEDB_ID_12345` are direct ID mappings. Retained internal subsets with renamed headers can recover IDs by exact sequence from the supplied original ID-bearing FASTAs. If several IDs match, all are kept and the ambiguity is recorded. Other records use exact-sequence API lookup.

Select the original `Ind-ibce.fasta` too if you want the notebook to recover IDs for renamed internal subsets. Original sequences/labels remain authoritative.


In [ ]:
def read_fasta(path):
    records, header, chunks = [], None, []
    def add():
        if header is None: return
        seq = "".join(chunks).replace(" ", "").upper()
        if not seq: raise ValueError(f"Empty FASTA record: {header} in {path}")
        records.append((header, seq))
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith(">"):
            add(); header, chunks = line[1:], []
        elif header is None: raise ValueError(f"Sequence before first header in {path}")
        else: chunks.append(line)
    add()
    if not records: raise ValueError(f"No records in {path}")
    return records

def infer_label(header):
    h = header.lower()
    if h.startswith(("positive", "pos")) or "|1|" in h: return "positive"
    if h.startswith(("negative", "neg")) or "|0|" in h: return "negative"
    return "unknown"

DATASETS, INPUT_MANIFEST = {}, []
for partition, path in [("training", TRAIN_PATH)] + [(f"test_{i+1:02d}", p) for i,p in enumerate(TEST_PATHS)]:
    raw = read_fasta(path)
    DATASETS[partition] = [{"partition": partition, "file_name": path.name, "record_index": i+1,
        "header": h, "sequence": seq, "label": infer_label(h),
        "header_iedb_id": int(re.search(r"IEDB_ID_(\d+)", h).group(1)) if re.search(r"IEDB_ID_(\d+)", h) else None}
        for i,(h,seq) in enumerate(raw)]
    INPUT_MANIFEST.append({"partition": partition, "path": str(path), "file_name": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "records": len(raw)})
    print(partition, path.name, "n=",len(raw), "length=",min(len(s) for h,s in raw),max(len(s) for h,s in raw),
          "unique_sequences=",len({s for h,s in raw}), "labels=",dict(Counter(infer_label(h) for h,s in raw)))
ALL_RECORDS = [r for a in DATASETS.values() for r in a]
SEQUENCE_TO_IDS = defaultdict(set)
for r in ALL_RECORDS:
    if r["header_iedb_id"] is not None: SEQUENCE_TO_IDS[r["sequence"]].add(r["header_iedb_id"])
for r in ALL_RECORDS:
    r["candidate_ids"] = [r["header_iedb_id"]] if r["header_iedb_id"] is not None else sorted(SEQUENCE_TO_IDS.get(r["sequence"], set()))
    r["lookup_route"] = "header_id" if r["header_iedb_id"] is not None else "sequence_crosswalk" if r["candidate_ids"] else "exact_sequence"
print("Unique numeric IDs:", len({i for r in ALL_RECORDS for i in r["candidate_ids"]}))
print("Records by initial lookup route:", dict(Counter(r["lookup_route"] for r in ALL_RECORDS)))
SMOKE_OK = False


## API client, retry policy and checkpoints
Sequential batches avoid a large simultaneous request load. Successful empty responses are cached as `not_found`; failed requests are recorded as `error` and retried on the next run. Incomplete pages are not cached as complete. API lookup never uses the positive/negative label as a search filter.


In [ ]:
def cache_path(kind, key): return CACHE_DIR / f"{kind}_{digest(str(key))}.json"
def load_cache(kind, key):
    p = cache_path(kind,key)
    if p.exists() and not REFRESH_CACHE:
        try:
            obj = json.loads(p.read_text())
            if obj.get("schema") == CACHE_SCHEMA and obj.get("status") != "error": return obj
        except (ValueError, OSError): pass
    return None

def save_cache(kind, key, rows, status="ok", error=None):
    obj = {"schema":CACHE_SCHEMA,"kind":kind,"key":str(key),"retrieved_at":now(),
           "status":status,"rows":rows,"error":error}
    atomic_json(cache_path(kind,key),obj)
    return obj

def request_page(params):
    url = API_URL + "?" + urllib.parse.urlencode(params)
    last = None
    for attempt in range(MAX_RETRIES):
        start = time.monotonic()
        try:
            req = urllib.request.Request(url, headers={"Accept":"application/json", "User-Agent":"IEDB-antigen-overlap-audit/1.0"})
            with urllib.request.urlopen(req,timeout=REQUEST_TIMEOUT) as response:
                rows = json.loads(response.read())
            if not isinstance(rows,list): raise ValueError("API did not return a JSON list")
            if any(not isinstance(r,dict) or "structure_id" not in r for r in rows): raise ValueError("Unexpected API record schema")
            REQUEST_LOG.append({"time":now(),"url":url,"attempt":attempt+1,"seconds":round(time.monotonic()-start,3),"rows":len(rows),"status":"ok"})
            time.sleep(PAUSE_SECONDS)
            return rows
        except Exception as exc:
            last = exc
            REQUEST_LOG.append({"time":now(),"url":url,"attempt":attempt+1,"status":"error","error":str(exc)})
            if isinstance(exc,urllib.error.HTTPError) and exc.code not in {408,413,414,429,500,502,503,504}: break
            delay = min(30,2**attempt)
            if isinstance(exc,urllib.error.HTTPError):
                retry_after = exc.headers.get("Retry-After", "")
                if retry_after.isdigit(): delay = min(30,max(delay,int(retry_after)))
            time.sleep(delay)
    raise RuntimeError(f"IEDB request failed after retries: {last}")

def query_all(filter_params):
    result, offset = [], 0
    for page in range(100):
        rows = request_page({**filter_params,"select":FIELDS,"order":"structure_id","limit":PAGE_SIZE,"offset":offset})
        result.extend(rows)
        if len(rows) < PAGE_SIZE: return result
        offset += PAGE_SIZE
    raise RuntimeError("Pagination safety limit reached; lookup is incomplete.")

def fetch_batch(kind, keys):
    keys = list(dict.fromkeys(keys))
    if not keys: return
    pending = [k for k in keys if load_cache(kind,k) is None]
    if not pending: return
    field = "structure_id" if kind == "id" else "linear_sequence"
    vals = ",".join(str(k) for k in pending)
    try:
        rows = query_all({field: f"in.({vals})"})
    except Exception as exc:
        # Split oversized/problematic requests before falling back to an error checkpoint.
        if len(pending)>1 and any(code in str(exc) for code in ["413", "414"]):
            mid = len(pending)//2
            fetch_batch(kind,pending[:mid]); fetch_batch(kind,pending[mid:]); return
        for key in pending: save_cache(kind,key,[],status="error",error=str(exc))
        raise RuntimeError("Batch retrieval failed; checkpoints saved. Check Internet/API and rerun.") from exc
    grouped = defaultdict(list)
    for row in rows:
        value = row.get("structure_id") if kind == "id" else (row.get("linear_sequence") or "").upper()
        grouped[str(value)].append(row)
    for k in pending:
        rr = grouped.get(str(k),[])
        save_cache(kind,k,rr,status="ok" if rr else "not_found")

def fetch_many(kind,keys,batch_size):
    keys = sorted(set(keys))
    for start in range(0,len(keys),batch_size):
        fetch_batch(kind,keys[start:start+batch_size])
        if start == 0 or (start//batch_size+1)%10==0 or start+batch_size>=len(keys):
            print(f"{kind}: {min(start+batch_size,len(keys))}/{len(keys)} keys checked",flush=True)
        atomic_json(OUTPUT_DIR/"request_log.json",REQUEST_LOG)

def cached_entry(kind,key):
    obj = load_cache(kind,key)
    if obj is not None: return obj
    p = cache_path(kind,key)
    return json.loads(p.read_text()) if p.exists() else {"status":"not_requested","rows":[]}

def id_resolution(record):
    matches, bad, statuses = [], [], []
    for ident in record["candidate_ids"]:
        obj = cached_entry("id",ident);statuses.append(obj["status"])
        for row in obj["rows"]:
            if (row.get("linear_sequence") or "").upper() == record["sequence"]: matches.append(row)
            else: bad.append(row.get("structure_id"))
    return matches,bad,statuses

def exact_allowed(sequence): return bool(sequence) and not (set(sequence)-set("ACDEFGHIKLMNPQRSTVWY"))

def resolve_record(record):
    matches,bad,statuses = id_resolution(record)
    fallback = not matches
    route = record["lookup_route"]
    if fallback and exact_allowed(record["sequence"]):
        obj = cached_entry("sequence",record["sequence"])
        matches = [r for r in obj["rows"] if (r.get("linear_sequence") or "").upper()==record["sequence"]]
        statuses.append(obj["status"]);route = "exact_sequence_fallback" if record["candidate_ids"] else "exact_sequence"
    unique = {str(r["structure_id"]):r for r in matches}
    return list(unique.values()),route,bad,statuses


## Offline tests (no Internet requests)
Checks wrapped FASTA parsing, missing labels, version handling and the overlap logic. These tests do not establish that every IEDB record can be resolved.


In [ ]:
def protein_keys(rows):
    keys = {"curated_exact":set(),"curated_version_normalized":set(),"parent_antigen":set()}
    for row in rows:
        for a in row.get("curated_source_antigens") or []:
            if not isinstance(a,dict): continue
            iri = (a.get("iri") or "").strip()
            acc = (a.get("accession") or "").strip()
            org = a.get("source_organism_iri") or a.get("source_organism_name") or "unknown_organism"
            if iri: keys["curated_exact"].add(iri)
            elif acc: keys["curated_exact"].add(f"ACCESSION:{acc}|{org}")
            if iri:
                namespace, sep, identifier = iri.partition(":")
                key = namespace + ":" + re.sub(r"\.\d+$","",identifier) if sep else iri
                keys["curated_version_normalized"].add(f"{key}|{org}")
            elif acc: keys["curated_version_normalized"].add("ACCESSION:" + re.sub(r"\.\d+$", "", acc) + "|" + str(org))
        parents = row.get("parent_source_antigen_iris") or []
        if isinstance(parents,str): parents = [parents]
        keys["parent_antigen"].update(str(x) for x in parents if x)
    return keys

# No namespace conversion or protein-name matching is used; equivalence across
# different accession systems can remain unresolved. Parent keys help group isoforms.
_a = [{"structure_id":1,"curated_source_antigens":[{"iri":"UNIPROT:P12345.1","source_organism_iri":"NCBITaxon:1"}],"parent_source_antigen_iris":["UNIPROT:P12345"]}]
_b = [{"structure_id":2,"curated_source_antigens":[{"iri":"UNIPROT:P12345.2","source_organism_iri":"NCBITaxon:1"}],"parent_source_antigen_iris":["UNIPROT:P12345"]}]
assert not protein_keys(_a)["curated_exact"] & protein_keys(_b)["curated_exact"]
assert protein_keys(_a)["curated_version_normalized"] & protein_keys(_b)["curated_version_normalized"]
assert protein_keys(_a)["parent_antigen"] & protein_keys(_b)["parent_antigen"]
assert not any(protein_keys([]).values())
assert infer_label("Negative_1") == "negative"
_tmp = OUTPUT_DIR / "_parser_test.fasta"
_tmp.write_text(">Positive_IEDB_ID_1\nACD\nEF\n>other\nGHIK\n")
assert read_fasta(_tmp)==[("Positive_IEDB_ID_1","ACDEF"),("other","GHIK")]
_tmp.unlink()
print("Offline tests passed.")


## Small live API test — run before the full audit
This requests at most three records from each selected file, plus known IEDB ID 161711. Missing matches are legitimate outcomes; network failures and unexpected sequence responses are surfaced separately. The displayed overlaps are only smoke-test previews, not final benchmark findings.


In [ ]:
SMOKE_OK = False
SMOKE_RECORDS = [r for group in DATASETS.values() for r in group[:3]]
start = time.monotonic()
fetch_many("id",[161711]+[i for r in SMOKE_RECORDS for i in r["candidate_ids"]],ID_BATCH_SIZE)
known = cached_entry("id",161711)
if known["status"] != "ok" or not any(r.get("linear_sequence")=="TLDVRMINIEASQLA" for r in known["rows"]):
    raise RuntimeError("Known-ID smoke test failed. Check Internet/API availability; inspect request_log.json.")
fallback_sequences = [r["sequence"] for r in SMOKE_RECORDS if not id_resolution(r)[0] and exact_allowed(r["sequence"])]
fetch_many("sequence",fallback_sequences,SEQUENCE_BATCH_SIZE)
failures = []
for r in SMOKE_RECORDS:
    rows,route,bad,statuses = resolve_record(r)
    if not rows and "error" in statuses: failures.append(r["header"])
    print(r["file_name"],r["header"].split()[0],"route=",route,"matches=",len(rows),
          "source_counts=",{k:len(v) for k,v in protein_keys(rows).items()},"ID_sequence_mismatches=",bad)
if failures: raise RuntimeError(f"API failures for smoke records: {failures}. Rerun after checking Internet.")
SMOKE_OK = True
print(f"SMOKE TEST PASSED in {time.monotonic()-start:.1f} seconds. Review mappings before enabling the full run.")


## Full metadata retrieval — deliberately disabled initially
After reviewing the smoke test, change the switch below to True and rerun this cell. Reruns use existing successful cache entries. An API error stops full-run report generation rather than silently implying independence. Partial cache remains available for resuming.


In [ ]:
RUN_FULL_AUDIT = True  # Change to True only after reviewing the smoke test.
FULL_AUDIT_COMPLETED = False
if RUN_FULL_AUDIT:
    if not SMOKE_OK: raise RuntimeError("Run and pass the small live smoke test first.")
    begin = time.monotonic()
    unique_ids = {i for r in ALL_RECORDS for i in r["candidate_ids"]}
    fetch_many("id",unique_ids,ID_BATCH_SIZE)
    # Exact-sequence fallback covers missing/mismatched ID records and external files.
    fallback_sequences = {r["sequence"] for r in ALL_RECORDS if not id_resolution(r)[0] and exact_allowed(r["sequence"])}
    fetch_many("sequence",fallback_sequences,SEQUENCE_BATCH_SIZE)
    errors = []
    for r in ALL_RECORDS:
        rows,route,bad,statuses = resolve_record(r)
        # Even partly resolved candidate sets with an API error remain incomplete.
        if "error" in statuses: errors.append((r["partition"],r["record_index"]))
    if errors:
        raise RuntimeError(f"{len(errors)} records have failed lookups. Cache is saved. Rerun this cell to retry; no final report generated.")
    FULL_AUDIT_COMPLETED = True
    print(f"Full retrieval completed in {(time.monotonic()-begin)/60:.1f} minutes.")
else:
    print("Full audit not executed. Review smoke-test output, set RUN_FULL_AUDIT=True, then rerun this cell.")


## Generate overlap reports
For each test version, report both record-weighted and unique-sequence counts. External duplicate sequences therefore do not silently inflate unique-sequence estimates.

A mapped peptide with no observed overlap is **not certified antigen-independent**: training mapping may be incomplete, isoforms may use different accessions, and external sequence-based provenance is only a candidate linkage. If any candidate antigen is shared, record it as possible overlap. Multiple epitope matches and multiple source proteins are flagged.


In [ ]:
def write_csv(path,rows,fields):
    with Path(path).open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)

if FULL_AUDIT_COMPLETED:
    mapping, edges, resolved = [], [], {}
    for r in ALL_RECORDS:
        rows,route,bad,statuses = resolve_record(r);keys = protein_keys(rows)
        resolved[(r["partition"],r["record_index"])] = (keys,route)
        rr = {k:r[k] for k in ["partition","file_name","record_index","header","sequence","label"]}
        rr.update(length=len(r["sequence"]), lookup_route=route,
            matched_iedb_ids=";".join(str(x["structure_id"]) for x in rows),
            mismatched_header_ids=";".join(map(str,bad)),
            mapping_status="mapped" if any(keys.values()) else "matched_epitope_no_source" if rows else "unresolved",
            multiple_epitope_matches=len(rows)>1, multiple_curated_sources=len(keys["curated_exact"])>1,
            multiple_parent_antigens=len(keys["parent_antigen"])>1,
            curated_exact=";".join(sorted(keys["curated_exact"])),
            curated_version_normalized=";".join(sorted(keys["curated_version_normalized"])),
            parent_antigen=";".join(sorted(keys["parent_antigen"])))
        mapping.append(rr)
        for level,kk in keys.items():
            for key in sorted(kk): edges.append({"partition":r["partition"],"file_name":r["file_name"],"record_index":r["record_index"],"sequence":r["sequence"],"level":level,"antigen_key":key,"lookup_route":route})
    fields = list(mapping[0])
    write_csv(OUTPUT_DIR/"record_mappings.csv",mapping,fields)
    write_csv(OUTPUT_DIR/"unresolved_or_ambiguous.csv",[r for r in mapping if r["mapping_status"]!="mapped" or r["multiple_epitope_matches"] or r["multiple_curated_sources"] or r["multiple_parent_antigens"] or r["mismatched_header_ids"]],fields)
    write_csv(OUTPUT_DIR/"antigen_memberships.csv",edges,["partition","file_name","record_index","sequence","level","antigen_key","lookup_route"])
    train_keys = {level:set() for level in ["curated_exact","curated_version_normalized","parent_antigen"]}
    for r in DATASETS["training"]:
        keys,_=resolved[("training",r["record_index"])]
        for level in train_keys: train_keys[level].update(keys[level])
    summaries,overlap_rows,shared_rows = [],[],[]
    for partition,records in DATASETS.items():
        if partition=="training": continue
        train_maps=[r for r in mapping if r["partition"]=="training"]
        local=[r for r in mapping if r["partition"]==partition]
        for level in train_keys:
            test_keys=set();mapped_records=[];shared_records=[]
            for r in records:
                keys,route=resolved[(partition,r["record_index"])];kk=keys[level];shared=kk & train_keys[level]
                test_keys.update(kk)
                if kk: mapped_records.append(r)
                if shared: shared_records.append(r)
                overlap_rows.append({"partition":partition,"file_name":r["file_name"],"record_index":r["record_index"],"header":r["header"],"sequence":r["sequence"],"label":r["label"],"level":level,
                    "status":"shared_candidate_antigen" if shared else "no_observed_overlap_mapped" if kk else "unresolved_at_this_level",
                    "shared_antigen_keys":";".join(sorted(shared)),"candidate_antigen_count":len(kk),"lookup_route":route})
            shared_keys=test_keys & train_keys[level]
            for key in sorted(shared_keys):shared_rows.append({"partition":partition,"file_name":records[0]["file_name"],"level":level,"antigen_key":key,
                "training_records":sum(key in resolved[("training",r["record_index"])][0][level] for r in DATASETS["training"]),
                "test_records":sum(key in resolved[(partition,r["record_index"])][0][level] for r in records)})
            n=len(records);mapped_n=len(mapped_records)
            summaries.append({"partition":partition,"file_name":records[0]["file_name"],"level":level,"total_records":n,"unique_sequences":len({r["sequence"] for r in records}),
                "positive_records":sum(r["label"]=="positive" for r in records),"negative_records":sum(r["label"]=="negative" for r in records),
                "mapped_records":mapped_n,"unresolved_records_at_level":n-mapped_n,"mapping_coverage_pct":100*mapped_n/n,
                "unique_test_antigens":len(test_keys),"unique_training_antigens":len(train_keys[level]),"shared_antigens":len(shared_keys),
                "test_records_with_possible_overlap":len(shared_records),"unique_test_sequences_with_possible_overlap":len({r["sequence"] for r in shared_records}),
                "overlap_pct_all_test_records":100*len(shared_records)/n,"overlap_pct_mapped_test_records":100*len(shared_records)/mapped_n if mapped_n else "",
                "training_records_mapped_at_level":sum(bool(resolved[("training",r["record_index"])][0][level]) for r in DATASETS["training"]),
                "test_records_multiple_epitope_matches":sum(r["multiple_epitope_matches"] for r in local),
                "test_records_with_id_sequence_mismatch":sum(bool(r["mismatched_header_ids"]) for r in local)})
    write_csv(OUTPUT_DIR/"overlap_summary.csv",summaries,list(summaries[0]))
    write_csv(OUTPUT_DIR/"test_record_overlap.csv",overlap_rows,list(overlap_rows[0]))
    write_csv(OUTPUT_DIR/"shared_antigens.csv",shared_rows,["partition","file_name","level","antigen_key","training_records","test_records"])
    atomic_json(OUTPUT_DIR/"run_manifest.json",{"generated_at":now(),"api":API_URL,"fields":FIELDS,"input_manifest":INPUT_MANIFEST,"python":sys.version,"platform":platform.platform(),
        "settings":{"id_batch_size":ID_BATCH_SIZE,"sequence_batch_size":SEQUENCE_BATCH_SIZE,"page_size":PAGE_SIZE,"pause_seconds":PAUSE_SECONDS,"timeout":REQUEST_TIMEOUT},
        "interpretation":"Exact-sequence lookup provides candidate provenance; unresolved mappings do not demonstrate independence. Curated exact keys retain accession versions and namespaces. Version-normalized curated keys remove accession suffix versions, retain namespace and organism. Parent keys use IEDB parent antigen IRIs."})
    atomic_json(OUTPUT_DIR/"request_log.json",REQUEST_LOG)
    print("OVERLAP SUMMARY")
    try:
        import pandas as pd
        display(pd.DataFrame(summaries))
    except ImportError:
        for row in summaries: print(row)
    print("Reports saved in",OUTPUT_DIR)
else:
    print("Final reports skipped: complete the full metadata audit first.")


## Download checkpoints or final results
You may run this cell after the smoke test or an interrupted retrieval to download the cache. The final ZIP is produced only after successful full-run report generation. Save notebook outputs and the archive before ending the Kaggle session.


In [ ]:
cache_zip=OUTPUT_DIR/"iedb_cache.zip"
with zipfile.ZipFile(cache_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CACHE_DIR.glob("*.json")): z.write(p,"cache/"+p.name)
display(FileLink(str(cache_zip)))
if FULL_AUDIT_COMPLETED and (OUTPUT_DIR/"overlap_summary.csv").exists():
    archive=OUTPUT_DIR/"iedb_antigen_audit_results.zip"
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT_DIR.rglob("*")):
            if p.is_file() and p.suffix != ".zip": z.write(p,str(p.relative_to(OUTPUT_DIR)))
    display(FileLink(str(archive)))
    print("Also download the archive from Kaggle's working/output file panel if the link is not clickable.")


## How to interpret the reports
- `overlap_summary.csv`: separate rows for every selected test file and mapping level; includes denominators and training mapping completeness.
- `record_mappings.csv`: sequence validation, lookup route, resolved IDs and antigens for every input record.
- `test_record_overlap.csv`: per-test-record shared-candidate / no-observed-overlap / unresolved status.
- `shared_antigens.csv`: shared protein identifiers and affected record counts.
- `unresolved_or_ambiguous.csv`: missing metadata, multiple mappings and ID/sequence discrepancies for review.
- `run_manifest.json`, `request_log.json` and cache JSONs: input hashes, dates, query parameters and reproducible responses.

Use parent-antigen grouping alongside precise source-protein accessions; neither automatically proves homology independence. Do not describe unresolved external records as antigen-disjoint, and do not silently infer the original negative-example protein from an unrelated exact IEDB match. Further source-paper provenance work may be necessary.
